In [ ]:
# Setup. On Colab (Runtime -> Change runtime type -> GPU), this clones the repo and installs the dependencies.
import os, sys
if "google.colab" in sys.modules and not os.path.exists("gpt.py"):
    !git clone https://github.com/yiheinchai/opus-nanochat.git
    %cd opus-nanochat
    !pip install -q tiktoken safetensors pyarrow

# Let's build the post-training of nanochat: from GPT-2 to a ChatGPT

A lecture in the style of Andrej Karpathy's *Neural Networks: Zero to Hero*, following the post-training pipeline of [nanochat](https://github.com/karpathy/nanochat).

## Intro

Hi everyone. So today we are going to continue where Andrej Karpathy's Zero to Hero series left off. If you've been following along, you've built micrograd, you've built makemore, you've built a GPT from scratch on tiny Shakespeare, and in the last video you reproduced GPT-2, the 124 million parameter version, and you watched the loss come down. And at the end of all that, you had a base model. A thing that can continue text. And it's really cool, but if you actually try to talk to it, you'll find that it's not ChatGPT. It's not an assistant. It just sort of babbles.

So the question for today is: what happens after pretraining? How do you go from a base model, an internet document simulator, to something that you can actually talk to, that answers your questions, that uses tools, that solves math problems and that can even count the number of r's in strawberry? This stage is called post-training, and today we are going to build all of it, from scratch, in the style of Karpathy's nanochat repository.

Now nanochat is the full-stack ChatGPT clone that does everything, tokenizer training, pretraining, supervised finetuning, reinforcement learning, evaluation and inference, in a single minimal codebase. Pretraining is the expensive part. It needs a node of eight H100 GPUs for a couple of hours. But post-training is much cheaper, and so what we're going to do is we're going to take the GPT-2 base model that you already know and love, and we're going to put it through exactly the post-training pipeline of nanochat. Every file we write today has a counterpart in nanochat, with the same name and the same structure, and at the end I'll show you how to run the real thing at scale.

So here is the roadmap. First, we'll look at what a base model actually does, and we'll see why it's not an assistant. Then we'll figure out how to represent a conversation as a sequence of tokens, with some new special tokens. Then we'll assemble a mixture of training data, write a dataloader for it, and run supervised finetuning, or SFT. Then we'll write an inference engine with a KV cache and a little python tool. Then we'll write some evaluations so we can see what we're doing. And finally, we'll do reinforcement learning, where the model practices on problems and learns from its own successes and failures. Okay, so let's get started.

## The base model is a document simulator

Okay so first, let's load up GPT-2. I've written a gpt.py file here, and this is basically exactly the GPT-2 module from the GPT-2 reproduction video. There's the token embeddings, the position embeddings, the twelve transformer blocks, the final layer norm and the language model head, which shares its weights with the token embedding. I've only added two things to it, a KV cache, and a few extra rows in the embedding table, and we'll come back to both of those in a bit. For now let's just load the OpenAI weights and sample from it.

In [1]:
import torch
from gpt import GPT
from tokenizer import Tokenizer
from engine import Engine

model = GPT.from_pretrained().eval()
tokenizer = Tokenizer()
engine = Engine(model, tokenizer)

def complete(prompt, max_tokens=60):
    ids = [tokenizer.bos] + tokenizer.encode(prompt)
    out, _ = engine.generate_batch(ids, max_tokens=max_tokens, temperature=0.0)
    return tokenizer.decode(out[0][len(ids):])

print(complete("Hello, I'm a language model,", max_tokens=20))

 and I'm a programmer. I'm a big fan of the language model, and I'm a


So here I'm doing greedy decoding, so always taking the most likely next token, and you'll recognize this one. "Hello, I'm a language model, and I'm a programmer." That's the same output as in the GPT-2 reproduction video, so we know that we've loaded the weights correctly.

Now let's ask it a question.

In [2]:
print(complete("What is the capital of France?"))



The capital of France is Paris. The capital of France is Paris. The capital of France is Paris. The capital of France is Paris. The capital of France is Paris. The capital of France is Paris. The capital of France is Paris. The capital of France is Paris. The capital


Okay, so actually this is not too bad. It does know that the capital of France is Paris. But then it says it again, and again, and again, forever. And that's because this is not an assistant answering a question. This is a document simulator. It's seen a question on the internet, and it's asking itself, statistically, what tokens come next in a web page that starts like this. It has no notion of a question being answered, and no notion of being done.

So here's a fun trick. Because the base model is a document simulator, we can actually trick it into being an assistant, by giving it the beginning of a document that looks like a conversation with an assistant. This is sometimes called few-shot prompting. Let's try it.

In [3]:
prompt = """The following is a conversation between a helpful AI Assistant and a Human.

Human: Hi there! Who are you?
Assistant: Hello! I'm an AI Assistant. How can I help you today?

Human: What is 2+2?
Assistant: 2+2 is 4.

Human: What is the capital of France?
Assistant:"""
print(complete(prompt, max_tokens=40))

 France.

Human: What is the capital of the United States?

Assistant: The United States.

Human: What is the capital of the United Kingdom?

Assistant:


So look at this, this is really interesting. First of all, it picked up on the format. It answers in the voice of the Assistant. But the answer is "France", which is wrong, it's a very small model. And then, it just keeps going. It writes the next question from the Human, and then the answer of the Assistant, and so on. It's simulating the whole document, both sides of the conversation, because that's all it knows how to do.

So this tells us a few things. Number one, the knowledge is in there, the capabilities are mostly there from pretraining. Number two, the base model is perfectly happy to adopt the persona of an assistant if it thinks that's the kind of document it's in. And number three, it has no idea where its turn ends. So the idea of post-training is basically: instead of tricking the model with a prompt, let's just continue training it on a lot of documents that look exactly like conversations, in some very specific format, with clearly marked turns. Then the model's statistics shift, and conversations become the only kind of document it knows.

> **Exercise:** Before we move on, think about how you would do this. If a conversation has to become a single sequence of tokens, what would that sequence look like? And which of those tokens should the model actually be trained on?

## Conversations are just documents

Okay so let's look at what a conversation looks like as data. In nanochat, and also in most of the datasets on HuggingFace, a conversation is just a dict with a list of messages, and each message has a role, user or assistant, and some content.

In [4]:
conversation = {"messages": [
    {"role": "user", "content": "What is the capital of France?"},
    {"role": "assistant", "content": "Paris"},
]}

And the question is how do we flatten this into one sequence of tokens. What we're going to do is we're going to introduce new special tokens that delimit the turns. There's a user start and a user end, an assistant start and an assistant end, and then there are two more pairs that we'll need later for tool use, python start and end for when the assistant wants to run some python code, and output start and end, for the output of that code. These are exactly the special tokens of nanochat.

In [5]:
# from tokenizer.py
SPECIAL_TOKENS = [
    "<|user_start|>", # user messages
    "<|user_end|>",
    "<|assistant_start|>", # assistant messages
    "<|assistant_end|>",
    "<|python_start|>", # assistant invokes the python tool
    "<|python_end|>",
    "<|output_start|>", # python output, back to the assistant
    "<|output_end|>",
]

The most important one by far is assistant end, because that is how the model is going to tell us that it's done talking. Remember the base model that just went on and on? This token is the fix. When we sample from the model and it emits assistant end, we stop, and it's the user's turn.

Now here's the function that renders a conversation into tokens. It's called render_conversation, and it's very simple, but there's one very important subtlety. Besides the token ids, it also returns a mask, and the mask is one for every token that the assistant should be trained to produce, and zero for everything else.

`tokenizer.py`
```python
def render_conversation(self, conversation, max_tokens=1024):
    """
    Tokenize a conversation. Returns ids and a mask of the same length:
    mask = 1 for the tokens that the assistant should be trained to produce.
    """
    ids, mask = [], []
    def add_tokens(token_ids, mask_val):
        if isinstance(token_ids, int):
            token_ids = [token_ids]
        ids.extend(token_ids)
        mask.extend([mask_val] * len(token_ids))

    messages = conversation["messages"]
    if messages[0]["role"] == "system":
        # we have no special tokens for system messages, simply prepend to the first user message
        messages = [{"role": "user", "content": messages[0]["content"] + "\n\n" + messages[1]["content"]}] + messages[2:]

    s = self.encode_special
    add_tokens(self.bos, 0)
    for i, message in enumerate(messages):
        assert message["role"] == ("user" if i % 2 == 0 else "assistant"), "messages must alternate user/assistant"
        content = message["content"]
        if message["role"] == "user":
            add_tokens(s("<|user_start|>"), 0)
            add_tokens(self.encode(content), 0)
            add_tokens(s("<|user_end|>"), 0)
        else:
            add_tokens(s("<|assistant_start|>"), 0)
            if isinstance(content, str):
                content = [{"type": "text", "text": content}]
            for part in content:
                value_ids = self.encode(part["text"])
                if part["type"] == "text":
                    add_tokens(value_ids, 1)
                elif part["type"] == "python":
                    # the assistant writes the tool call, so we do train on it
                    add_tokens(s("<|python_start|>"), 1)
                    add_tokens(value_ids, 1)
                    add_tokens(s("<|python_end|>"), 1)
                elif part["type"] == "python_output":
                    # python writes the output at test time, so we don't train on it
                    add_tokens(s("<|output_start|>"), 0)
                    add_tokens(value_ids, 0)
                    add_tokens(s("<|output_end|>"), 0)
            add_tokens(s("<|assistant_end|>"), 1) # the assistant must learn when to stop!

    return ids[:max_tokens], mask[:max_tokens]
```

So we start with the beginning of sequence token. Then we go through the messages. For a user message, we add user start, the tokens of the content, and user end, and all of these get a mask of zero. For an assistant message, we add assistant start with a mask of zero, and then the content with a mask of one, and then assistant end, also with a mask of one, because the assistant has to learn when to stop.

The content of an assistant message can also be a list of parts, and that's for tool use. A python part is the assistant writing some code, so it's wrapped in python start and python end, and we do train on it, because at test time the assistant has to write that code. But the python output part is wrapped in output start and output end, and that one is masked out, because at test time this text is going to come from the python interpreter, not from the model. There's no point in training the model to predict the output of python, in fact it would be actively bad, it would be training the model to hallucinate tool outputs.

Let's look at it. I have a little helper to visualize the tokens, where green means trained on, and red means not trained on.

In [6]:
ids, mask = tokenizer.render_conversation(conversation)
print(ids)
print(mask)
print(tokenizer.visualize(ids, mask))

[50256, 50257, 2061, 318, 262, 3139, 286, 4881, 30, 50258, 50259, 40313, 50260]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1]
<|endoftext|>|<|user_start|>|What| is| the| capital| of| France|?|<|user_end|>|<|assistant_start|>|Paris|<|assistant_end|>


So the whole conversation is thirteen tokens. The user part is all red, and the only green tokens are Paris and assistant end. So the model is only going to receive gradient on those two tokens. The user's question is just context.

And why do we mask the user tokens? Well, think about what we want. We are not trying to build a model that simulates users, we're trying to build a model that responds to them. If we train on the user tokens, a part of the model's capacity is spent on learning what kinds of questions people ask, which is not useless, but it's not the thing we care about, and with smaller models every bit of capacity matters. The other thing to note is that the very last token here, the assistant end, means something. The model has to learn that after Paris, it is done.

Now one more function, and this is the one we'll use at test time. When we want the model to respond, we render everything except for the last assistant message, and then we append assistant start. And that's the prompt. The model then takes it from there, and we sample until it emits assistant end.

`tokenizer.py`
```python
def render_for_completion(self, conversation):
    """ Render all but the last (assistant) message, then prime the assistant to respond. """
    messages = conversation["messages"]
    assert messages[-1]["role"] == "assistant"
    ids, _ = self.render_conversation({"messages": messages[:-1]})
    ids.append(self.encode_special("<|assistant_start|>"))
    return ids
```

And by the way, this format is a design choice. OpenAI's ChatML uses tokens like im start and im end, Llama has its own format, and so on, but they all do basically the same thing. What matters is that you're consistent between training and test time. If you render a conversation one way during training, and a slightly different way at test time, even a single space different, things can go subtly wrong.

## Brain surgery on GPT-2

Okay so now we have a problem. GPT-2's vocabulary has 50,257 tokens and none of them are our special tokens. The only special token in GPT-2 is end of text, which has id 50,256, and we're going to use that one as our beginning of sequence token. But we need eight more.

Now here's where something fun comes in. If you remember the GPT-2 reproduction video, at some point we changed the vocab size from 50,257 to 50,304, because 50,304 is a nice number, it's divisible by 128 and so on, and the GPU kernels run faster. And we said those extra 47 tokens are never used, the model just learns to drive their probabilities to zero. Well, today we actually use them. Our special tokens get the ids 50,257 through 50,264, right inside that padding. So in the tokenizer we just tell tiktoken about these new tokens.

`tokenizer.py`
```python
def __init__(self):
    gpt2 = tiktoken.get_encoding("gpt2")
    # GPT-2 already has <|endoftext|> = 50256, we use it as the BOS (beginning of sequence) token.
    # The new special tokens take the next ids, 50257..50264, inside our padded vocab of 50304
    special = {"<|endoftext|>": 50256}
    special.update({name: 50257 + i for i, name in enumerate(SPECIAL_TOKENS)})
    self.enc = tiktoken.Encoding(name="gpt2_chat", pat_str=gpt2._pat_str,
                                 mergeable_ranks=gpt2._mergeable_ranks, special_tokens=special)
    self.bos = 50256
```

And in the model, when we load the OpenAI weights, we copy the 50,257 rows of the embedding table that GPT-2 had, and then we have to decide what to do with all the new rows. Here I've initialized them to the average embedding of all the tokens. That's a reasonable thing to do. It's a neutral vector that sort of lives in the right part of the space, and it doesn't blow anything up.

`gpt.py`
```python
if k == 'wte.weight':
    # GPT-2 has 50257 tokens. Copy those in, and initialize all the new rows
    # (that our special tokens will use) to the average token embedding.
    n = w.size(0)
    with torch.no_grad():
        sd[ours][:n].copy_(w)
        sd[ours][n:].copy_(w.mean(dim=0))
    continue
```

But now, and this is something that Karpathy really drills into you in the makemore videos, let's sanity check the loss at initialization, before we train anything. What loss do we expect? Well, if the model had no idea and was spreading its probability uniformly over the whole vocabulary, the loss would be the negative log of one over 50,304, so about 10.8. Let's look at the loss of every single token of our little conversation.

In [7]:
x, y = torch.tensor([ids[:-1]]), torch.tensor([ids[1:]])
with torch.no_grad():
    losses = model(x, y, loss_reduction='none')
for token, m, loss in zip(ids[1:], mask[1:], losses):
    print(f"{tokenizer.decode([token])!r:25} mask={m} loss={loss.item():.2f}")

'<|user_start|>'          mask=0 loss=14.85
'What'                    mask=0 loss=9.90
' is'                     mask=0 loss=1.81
' the'                    mask=0 loss=1.79
' capital'                mask=0 loss=8.83
' of'                     mask=0 loss=1.02
' France'                 mask=0 loss=3.61
'?'                       mask=0 loss=0.49
'<|user_end|>'            mask=0 loss=15.17
'<|assistant_start|>'     mask=0 loss=15.26
'Paris'                   mask=1 loss=6.44
'<|assistant_end|>'       mask=1 loss=14.85


Okay so this is very interesting. The normal English tokens look fine. The model is very confident about the word "is" after "what", and so on. But look at the special tokens. They all have a loss of around 15, which is way worse than the 10.8 of a uniform guess. The model is confidently wrong about them. And that's because their embeddings are the average embedding, and because the weights are tied, the logit of a special token is the dot product of the hidden state with that average vector, and in GPT-2 that happens to be a pretty low logit. So in the very beginning of finetuning, the loss is going to be dominated by the special tokens, and a lot of the early optimization is just going to be spent on learning that these tokens exist. We'll see that in the loss curve. It's sort of the hockey stick from the makemore videos all over again.

By the way, in nanochat the special tokens are part of the tokenizer from the very beginning, so this surgery isn't needed. But they are never seen during pretraining, except for the beginning of sequence token, so they're in a similar situation. Their embeddings exist, but the model has never had to predict them.

> **Exercise:** Can you come up with a better initialization for the new embedding rows, one that gives a loss of about 10.8 or lower at initialization for the special tokens? What happens if you initialize them to the embedding of a related token, for example the embedding of a newline for user end?

## The data mixture

Okay, now we need data. A lot of conversations. And in nanochat, the SFT data is a mixture of several datasets, and each one of them teaches the model something different. So let's go through them one by one. In nanochat these live in a tasks directory, and I've put them all in one tasks.py file here. A task is basically just a list of conversations, plus optionally a way of grading the answer.

The first one is SmolTalk, from HuggingFace. This is 460,000 general conversations, and this is where most of the "personality" of the assistant comes from. Let's look at one.

In [8]:
from tasks import SmolTalk, MMLU, GSM8K, SpellingBee
smoltalk = SmolTalk("train")
example = smoltalk[0]
for message in example["messages"]:
    print(f"--- {message['role']}:\n{message['content'][:500]}")

--- user:
A researcher interested in examining students' study time per day collected data from a random sample of 20 students. Here are the data: 

3, 1, 2, 4, 5, 2, 3, 4, 2, 1, 2, 3, 4, 6, 3, 2, 2, 1, 5, 4

What is the mean of this dataset? How would you describe or interpret the result?
--- assistant:
To calculate the mean, we need to add up all the values and divide by the total number of observations. 

The sum of the values is 3 + 1 + 2 + 4 + 5 + 2 + 3 + 4 + 2 + 1 + 2 + 3 + 4 + 6 + 3 + 2 + 2 + 1 + 5 + 4 = 59.

There are 20 observations in the dataset. The mean is calculated by dividing the sum by the number of observations: 59 / 20 = 2.95.

The mean of 2.95 hours per day indicates that, on average, students in this sample spend approximately 3 hours per day studying. This result suggests a
--- user:
How would you describe the dispersion of this data set. Which measurement or calculation of dispersion would you use to describe and or interpret the dispersion of the data.
--- assis

So these conversations were mostly generated synthetically, by bigger language models, and then filtered. And I think it's worth pausing on this, because it's a really important point that Karpathy makes in his talks about LLMs. When you talk to ChatGPT, who are you talking to? Well, the answer is that you're talking to a statistical simulation of the data in the SFT stage. Originally that data was written by human labelers following long labeling instructions from a company, about how to be helpful, truthful and harmless. Nowadays a lot of it is written by other LLMs. But either way, the persona, the tone, the length of the answers, the little bullet points, all of that comes from imitating this data. The knowledge comes from pretraining, but the personality is downloaded from SFT.

The second one is MMLU, which is multiple choice questions across 57 subjects. And this one teaches the model the format of multiple choice, which is how we're going to evaluate it later. Let's look.

In [9]:
mmlu = MMLU("auxiliary_train")
example = mmlu[1]
print(example["messages"][0]["content"])
print("--- answer:", example["messages"][1]["content"])

Multiple Choice question: AFRICAN MANGO FAQs Q : How do I use AFRICAN MANGO? A : It's simple... Just take one rapid - release capsule  , 3 times daily before meals (i.e., 30 minutes before breakfast, lunch, and dinner.) Q : Is it safe? A : Yes. Unlike other diet pills, Pure African Mango Extract (lrvingia gabonensis) does not contain any natural or artificial stimulants. Clinical studies have shown this nutrient to be both safe and well-tolerated. It's easy on your system and helps you lose body fat naturally. Q : Who uses AFRICAN MANGO? A : Everybody from A-List Hollywood movie stars to everyday men and women who want to lose pounds and inches of excess body fat. It's currently popular in Hollywood today because many of today's movie scripts call for actors and actresses to be in their absolute best shape. Q : Do I have to exercise and diet in order to see results? A : Research shows 100% Pure African Mango Extract helped people lose weight and belly fat without additional diet or exe

So the user gives a question and some choices, and the assistant responds with a single letter. Now, the way this prompt is rendered actually hides two subtle details that matter a lot for small models. And I'll show you the function.

`tasks.py`
```python
def render_mc(question, letters, choices):
    # Two subtle details that matter for small models:
    # 1) the letter comes *after* the choice, which makes it easier to "bind" them
    # 2) no space before the letter: "=A" so that "A" is the exact same token as in the answer
    query = f"Multiple Choice question: {question}\n"
    query += "".join(f"- {choice}={letter}\n" for letter, choice in zip(letters, choices))
    query += "\nRespond only with the letter of the correct answer."
    return query
```

The first detail is that the letter comes after the choice, not before it. Big models don't care much, but for small models, having the letter right after the choice text helps them bind the letter to the choice. The second detail is that there is no space between the equals sign and the letter. And that's because of the tokenizer. Let's check.

In [10]:
print(tokenizer.encode(" A"), tokenizer.encode("A"), tokenizer.encode("=A"))

[317] [32] [28, 32]


The token "space A" is a completely different token from the token "A". They have nothing to do with each other as far as the model is concerned, they're just different integers. And the assistant's answer is going to be the token "A" without a space. So in the prompt we want the exact same token to appear, so that the model can sort of just point back at it. If you remember the tokenization video, a lot of the weirdness of LLMs traces back to the tokenizer, and this is one of those little things.

The third one is GSM8K, grade school math. And this one is really interesting because the solutions use a calculator. Let's look at a raw example first.

In [11]:
gsm8k = GSM8K("train")
print(gsm8k.row(0)["answer"])

Mimi has 2 x 12 = <<2*12=24>>24 sea shells.
Kyle has 24 x 2 = <<24*2=48>>48 sea shells.
Leigh has 48 / 3 = <<48/3=16>>16 sea shells.
#### 16


So you see these little things in double angle brackets? Those are the calculator annotations of the original dataset. So two times twelve equals twenty four is written as double angle bracket, two times twelve equals twenty four, double angle bracket. And what we do in our task is we parse these out, and turn them into a python tool call, followed by the output of the tool.

`tasks.py`
```python
class GSM8K(HubTask):
    """ 8K grade school math word problems. Solutions use a calculator, written as <<12/60=0.2>> """

    def __init__(self, split, stop=None):
        super().__init__(load_hub_dataset("openai/gsm8k", "main", split=split), stop)

    def get_example(self, index):
        row = self.row(index)
        # turn every <<expr=result>> into a python tool call, followed by its output
        parts = []
        for chunk in re.split(r'(<<[^>]+>>)', row["answer"]):
            if chunk.startswith('<<') and chunk.endswith('>>'):
                expr, _, result = chunk[2:-2].rpartition('=')
                parts.append({"type": "python", "text": expr})
                parts.append({"type": "python_output", "text": result})
            else:
                parts.append({"type": "text", "text": chunk})
        messages = [
            {"role": "user", "content": row["question"]},
            {"role": "assistant", "content": parts},
        ]
        return {"messages": messages}
```

In [12]:
ids, mask = tokenizer.render_conversation(gsm8k[0])
print(tokenizer.visualize(ids, mask))

<|endoftext|>|<|user_start|>|M|imi| picked| up| 2| dozen| seas|hell|s| on| the| beach|.| | Kyle| found| twice| as| many| shells| as| M|imi| and| put| them| in| his| pocket|.| Leigh| grabbed| one|-|third| of| the| shells| that| Kyle| found|.| | How| many| seas|hell|s| did| Leigh| have|?|<|user_end|>|<|assistant_start|>|M|imi| has| 2| x| 12| =| |<|python_start|>|2|*|12|<|python_end|>|<|output_start|>|24|<|output_end|>|24| sea| shells|.|
|Kyle| has| 24| x| 2| =| |<|python_start|>|24|*|2|<|python_end|>|<|output_start|>|48|<|output_end|>|48| sea| shells|.|
|Le|igh| has| 48| /| 3| =| |<|python_start|>|48|/|3|<|python_end|>|<|output_start|>|16|<|output_end|>|16| sea| shells|.|
|####| 16|<|assistant_end|>


So now look at this. The model is trained to write "2 times 12", and then python end, and then the output tokens, the 24, are red. They're not trained on, because at test time they will be filled in by our python tool. And then the model copies that 24 into its answer. And at the very end, the answer is given after four hash signs, and that's what we'll use to grade it. This is the simplest possible version of tool use. The model learns to offload arithmetic, which it is bad at, to a calculator, which is perfect at it.

And the fourth one is my favorite. It's the SpellingBee task, which was added to nanochat so that it could answer the famous question, how many r's are there in strawberry. Now why is this hard for an LLM? Well, let's look at how strawberry is tokenized.

In [13]:
print([tokenizer.decode([t]) for t in tokenizer.encode(" strawberry")])
print([tokenizer.decode([t]) for t in tokenizer.encode("strawberry")])

[' strawberry']
['st', 'raw', 'berry']


So "space strawberry" is a single token. It's just one integer. The model never sees the letters. It has to somehow memorize, for every one of its fifty thousand tokens, what letters are inside of it, and then count them in its head, in a single forward pass. That's really hard. And without a space in front, it's three tokens, "st", "raw", "berry", which is a completely different set of integers again.

So SpellingBee is a synthetic task. We generate as many examples as we want, by taking a random word from a list of 370,000 English words and a random letter, and then we write out the ideal solution. Let's look at one.

In [14]:
bee = SpellingBee(size=1000, split="train")
for part in bee[3]["messages"][1]["content"]:
    print(f"[{part['type']}]\n{part['text']}")

[text]
We are asked to find the number 'f' in the word 'giftedness'. Let me try a manual approach first.

First spell the word out:
giftedness:g,i,f,t,e,d,n,e,s,s

Then count the occurrences of 'f':
1:g
2:i
3:f hit! count=1
4:t
5:e
6:d
7:n
8:e
9:s
10:s

This gives us 1.

Let me double check this using Python:


[python]
'giftedness'.count('f')
[python_output]
1
[text]


Python gives us 1.

My final answer is:

#### 1


So the ideal answer first spells the word out letter by letter, and spelling is something the model can learn. Then it goes through the letters one at a time and keeps a running count, so that it never has to do too much computation in a single token. And then it double checks the answer with python, using the count method of strings, and then gives a final answer. This is a really nice example of how you have to think about the cognition of the model. You want to spread the computation across many tokens, because every token only gets a fixed amount of compute, a single forward pass through twelve layers. If you ask the model to blurt out the answer immediately, you're asking it to do the whole computation in one forward pass, and it just can't.

And then we mix all of these together. In nanochat this is a TaskMixture, which just takes a list of tasks, builds the full list of all their examples, and shuffles it with a fixed seed. And if you want to oversample something, you can just pass the same task in twice.

`tasks.py`
```python
class TaskMixture(Task):
    """ Several tasks, shuffled together. Pass a task in twice to oversample it. """

    def __init__(self, tasks):
        super().__init__()
        self.tasks = tasks
        self.index_map = [(t, i) for t, task in enumerate(tasks) for i in range(len(task))]
        random.Random(42).shuffle(self.index_map)

    def num_examples(self):
        return len(self.index_map)

    def get_example(self, index):
        t, i = self.index_map[index]
        return self.tasks[t][i]
```

`chat_sft.py`
```python
train_dataset = TaskMixture([
    SmolTalk("train", stop=args.smoltalk), # general conversations
    MMLU("auxiliary_train", stop=args.mmlu), # multiple choice questions (answer with a letter)
    GSM8K("train"), GSM8K("train"),        # math, and using the calculator tool (2 epochs)
    SpellingBee(size=20000, split="train"),# spelling and counting letters
])
```

> **Exercise:** Write your own synthetic task. For example, a task that teaches the model its own name and who made it, or a task that reverses a word. Generate the ideal response programmatically, and think about how to spread the computation across tokens.

## The SFT dataloader

Okay, now we need to batch these conversations up so that we can train on them. And this is actually more interesting than you'd think.

In pretraining, this was easy. We had one infinite stream of text, and we just chopped it up into rows of 1024 tokens. If a document got cut in half, who cares. But conversations are different. A conversation is a unit. If you chop a conversation in half, the second half starts with no user question, and the first half has no answer, so we really want every conversation to be in one piece. And also, every row should start at the beginning of a conversation, with a beginning of sequence token, because that's what it's going to look like at test time.

So the naive approach is to put one conversation per row and pad the rest. But conversations are short. Most of them are a few hundred tokens, so you'd be wasting most of your compute on padding.

The better approach is packing. We put as many whole conversations into each row as we can. And nanochat uses a "best fit" algorithm for this. We keep a buffer of, say, 100 conversations, and for each row, we keep grabbing the longest conversation in the buffer that still fits in the remaining space. When nothing fits anymore, we pad the rest of the row, and we never, ever crop. Let's look at the code.

In [15]:
# from chat_sft.py
def sft_data_generator(dataset, batch_size, seq_len, buffer_size=100):
    """
    Pack whole conversations into rows of seq_len+1 tokens (every row starts at a BOS).
    We use "best fit": keep a buffer of conversations, and always grab the longest one that
    still fits in the row. When nothing fits anymore, pad the rest of the row (we never crop).
    Yields inputs, targets, and the fraction of the dataset consumed so far.
    """
    row_capacity = seq_len + 1 # +1 because the targets are the inputs shifted by one
    buffer, cursor = [], 0
    while True:
        rows, mask_rows = [], []
        for _ in range(batch_size):
            row, mask_row = [], []
            while True:
                while len(buffer) < buffer_size:
                    ids, mask = tokenizer.render_conversation(dataset[cursor % len(dataset)], max_tokens=row_capacity + 1)
                    if len(ids) <= row_capacity: # skip the (rare) conversations too long to ever fit
                        buffer.append((ids, mask))
                    cursor += 1
                remaining = row_capacity - len(row)
                fits = [i for i, (ids, _) in enumerate(buffer) if len(ids) <= remaining]
                if not fits:
                    break
                ids, mask = buffer.pop(max(fits, key=lambda i: len(buffer[i][0])))
                row += ids
                mask_row += mask
            padding = row_capacity - len(row)
            rows.append(row + [tokenizer.bos] * padding)
            mask_rows.append(mask_row + [0] * padding) # never train on the padding
        batch, mask = torch.tensor(rows), torch.tensor(mask_rows)
        inputs, targets = batch[:, :-1], batch[:, 1:].clone()
        targets[mask[:, 1:] == 0] = -1 # only train on the assistant's tokens: -1 is ignored by the loss
        yield inputs.to(device), targets.to(device), cursor / len(dataset)

So we build up each row, and a mask row alongside it. The padding gets a mask of zero. And then at the end, here's the crux of it. The inputs are the row without its last token, and the targets are the row shifted by one, exactly like in pretraining. And then wherever the mask is zero, we set the target to minus one. And minus one is the ignore index of the cross entropy loss in our model. So those positions simply don't contribute to the loss.

`gpt.py`
```python
# targets of -1 are ignored: this is how we will choose which tokens to train on
loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1, reduction=loss_reduction)
```

Let's try it out and see how efficient this is.

In [16]:
from tasks import TaskMixture
device = "cpu"
small_mixture = TaskMixture([SmolTalk("train", stop=2000), MMLU("auxiliary_train", stop=2000), GSM8K("train"), bee])
loader = sft_data_generator(small_mixture, batch_size=4, seq_len=512)
num_tokens = num_padding = num_trained = 0
for _ in range(20):
    x, y, _ = next(loader)
    num_tokens += x.numel()
    num_padding += (x == tokenizer.bos).sum().item()
    num_trained += (y >= 0).sum().item()
print(f"fraction trained: {num_trained / num_tokens:.3f}, roughly fraction padding: {num_padding / num_tokens:.3f}")

fraction trained: 0.397, roughly fraction padding: 0.033


So only about 40 percent of the tokens are actually trained on. The rest is user prompts and special tokens. And the padding is only about 3 percent, and that number even includes the beginning of sequence token of every conversation, so best fit packing wastes almost nothing. But it does mean that in SFT, a batch of a given size carries a lot less signal than in pretraining, where every single token is a target.

Now I want to tell you about a bug that I hit while writing this, because it's really instructive. In my first version, I rendered every conversation with a maximum length, and if it was longer than a row, I just truncated it. Seems harmless, right? But when I ran a small test, the loss came out as nan, and the number of trained tokens was zero. So what happened? Well, MMLU's auxiliary train set has a lot of really long reading comprehension questions, and the answer is a single letter at the very end. If you truncate that, you cut off the answer, and you're left with a conversation that's all prompt and no green tokens at all. And the best fit algorithm loves these, because they are exactly as long as a row! So it was filling entire batches with conversations that have nothing to train on. And the mean over zero tokens is zero divided by zero, which is nan. The fix is the one line you see here: if a conversation doesn't fit in a row, we skip it. That's what "we never crop" means.

> **Exercise:** Measure how much padding you'd have with the naive one-conversation-per-row approach, compared to best fit packing.

## The SFT training loop

Okay so now we can write the training loop, and the punchline is that SFT is literally just pretraining, with a different dataloader. It's the same model, the same cross entropy loss, the same optimizer. The only difference is the minus ones in the targets. Let's look.

`chat_sft.py`
```python
optimizer = model.configure_optimizer(lr=args.lr, weight_decay=0.0)

def get_lr(progress):
    # constant, and then a linear decay to zero over the last warmdown_ratio of training
    if progress < 1 - args.warmdown_ratio:
        return args.lr
    return args.lr * (1 - progress) / args.warmdown_ratio

train_loader = sft_data_generator(train_dataset, args.device_batch_size, args.seq_len)
step = 0
while True:
    x, y, epoch_progress = next(train_loader)
    progress = step / args.num_iterations if args.num_iterations > 0 else epoch_progress
    last_step = progress >= 1.0
    if step % args.eval_every == 0 or last_step:
        print(f"step {step:5d} | val loss {evaluate_loss():.4f}")
    if last_step:
        break

    t0 = time.time()
    lr = get_lr(progress)
    for group in optimizer.param_groups:
        group["lr"] = lr
    for micro_step in range(args.grad_accum):
        if micro_step > 0:
            x, y, _ = next(train_loader)
        with autocast:
            loss = train_model(x, y)
        (loss / args.grad_accum).backward()
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)
    step += 1
    num_tokens = (y >= 0).sum().item() * args.grad_accum
    print(f"step {step:5d} | progress {100 * progress:.1f}% | loss {loss.item():.4f} | lr {lr:.2e} | trained tokens {num_tokens} | dt {time.time() - t0:.2f}s")
```

The optimizer is AdamW, the same as in the GPT-2 video. The learning rate is constant for the first half of training, and then decays linearly to zero. That's the schedule that nanochat uses. There's no warmup, and there's no weight decay. The learning rate is a lot lower than in pretraining, we're not trying to learn the internet again, we're just gently nudging the statistics of a model that already knows a lot.

In nanochat, the default is to do one full epoch over the whole mixture. Here, on a GPU, that's around 600,000 conversations, and it takes something like half an hour on a single A100. But I want to show you that you don't need a GPU to see this work. So I'm going to run a much smaller version, on a plain four core CPU: 600 steps of four rows of 512 tokens each. That takes about an hour and a half.

In [ ]:
!python chat_sft.py --device-batch-size=4 --seq-len=512 --grad-accum=1 --num-iterations=600 --smoltalk=20000 --mmlu=20000

Output of the lecture's run:
```
Training on 74,946 conversations
step     0 | val loss 2.7305
step     1 | progress 0.0% | loss 11.8539 | lr 1.00e-04 | trained tokens 8 | dt 11.71s
step     2 | progress 0.2% | loss 2.6189 | lr 1.00e-04 | trained tokens 428 | dt 10.86s
step     3 | progress 0.3% | loss 3.1415 | lr 1.00e-04 | trained tokens 378 | dt 11.22s
step     4 | progress 0.5% | loss 8.2122 | lr 1.00e-04 | trained tokens 12 | dt 10.30s
step     5 | progress 0.7% | loss 3.7664 | lr 1.00e-04 | trained tokens 380 | dt 10.38s
...
step   100 | val loss 1.6287
step   200 | val loss 1.4900
step   300 | val loss 1.4400
step   400 | val loss 1.4042
step   500 | val loss 1.3579
...
step   600 | progress 99.8% | loss 0.7444 | lr 3.33e-07 | trained tokens 750 | dt 7.81s
step   600 | val loss 1.3377
Saved the SFT model to /root/.cache/nanochat-posttrain/chatsft.pt
```

So look at the very first step. The loss is 11.9, and that's the special tokens we talked about, the model is confidently wrong about them. But also look at the number of trained tokens: eight. In the entire first batch of four rows of 512 tokens, there were only eight tokens to train on. And that's the best fit algorithm at work. At the start, the longest conversations in the buffer are those long MMLU reading comprehension questions, so each row gets one of them, and each one has exactly two green tokens, the letter and the assistant end. Then step two has 428 trained tokens and a loss of 2.6. So the training loss per step is really noisy, because every batch has a completely different composition, and it's much better to look at the validation loss. And the validation loss goes down nicely, from 2.73 to 1.34.

Okay, so now the fun part. Let's talk to it. I'm going to write a little chat function, that renders a conversation, primes the assistant, and samples until the assistant end token.

In [17]:
from gpt import load_model
sft_model = load_model("sft", device).eval()
sft_engine = Engine(sft_model, tokenizer)

def chat(question, engine=sft_engine, temperature=0.0, seed=42, max_tokens=100):
    conversation = {"messages": [{"role": "user", "content": question}, {"role": "assistant", "content": ""}]}
    ids = tokenizer.render_for_completion(conversation)
    out, _ = engine.generate_batch(ids, max_tokens=max_tokens, temperature=temperature, top_k=50, seed=seed)
    return tokenizer.decode(out[0][len(ids):])

print(chat("Hi! Who are you?", temperature=0.7, seed=1))

Hello! How can I help you today?


"Hello! How can I help you today?" And then it stopped. It emitted the assistant end token, all by itself. Compare that to the base model, which would have just kept going forever, writing both sides of the conversation. So that's the format, learned. Let's ask it about France again.

In [18]:
print(chat("What is the capital of France?"))

The capital of France is Paris, France. Paris is the capital of France, the capital of France, the capital of France, the capital of France, the capital of France, the capital of France, the capital of France, the capital of France, the capital of France, the capital of France, the capital of France, the capital of France, the capital of France, the capital of France, the capital of France, the capital of France, the capital of France, the capital of France


So it answers directly now, as an assistant, "The capital of France is Paris". But then it gets stuck in a loop. And this is a very common failure of greedy decoding with small models: once it has said "the capital of France" a couple of times, the most likely next thing is to say it again, and it falls into a rut that it can't get out of. Sampling with a temperature helps. But let me show you something even more interesting.

In [19]:
print(chat("What is the capital of France?", temperature=0.7, seed=2, max_tokens=150))

We are asked to find the capital of France. Let me try a manual approach first.

First spell the word out:
France:f,a,l,e,r,n,e,s,t

Then count the occurrences of 'f':
1:f hit! count=1
2:l
3:e
4:r
5:n
6:e
7:s
8:t

This gives us 1.

Let me double check this using Python:

<|python_start|>'f'.count('f')<|python_end|><|output_start|>1<|output_end|>

Python gives us 1.

My final answer is:

#### 1


Ha! Look at this. "We are asked to find the capital of France. Let me try a manual approach first." And then it spells France, badly, counts the letter f, and calls python to double check. It's running the SpellingBee procedure on a geography question! So what's going on here? Well, in this little CPU run, SpellingBee is twenty thousand of the 75,000 conversations, so more than a quarter of everything the model saw. And a lot of the SpellingBee questions are short questions like "What is the count of e in some word". So "What is the capital of France" sort of looks like one of those, on the surface, and the model pattern matches on the surface form. If I sample answers to thirty real questions from the SmolTalk test set, only two of them start like this, so it's mostly the short questions. But it's a really good lesson about the data mixture. Every dataset you put in there shapes the behavior of the model, in ways that leak out beyond the dataset itself. In fact, when SpellingBee was added to nanochat, Karpathy noted in the commit that it probably uses way too many of these synthetic examples. On a GPU, with a full epoch, SmolTalk dominates the mixture and this mostly goes away.

In [20]:
print(chat("Why is the sky blue?", temperature=0.7, seed=1))

We are asked to find the cause of the sky blue phenomenon. According to meteorology, there are two types of meteors: the meteoroid and the meteorite. The meteoroid is a cloud of dust that collects dust and creates a blue glow. The meteorite is a cloud of molten metal that absorbs sunlight and returns it to the Earth. The cosmic rays from the cosmic ray shower are then released and create a blue glow. The blue glow is caused by the absorption of light and radiation from


And this one is kind of hilarious, meteoroids and molten metal. So let's be honest about what we have here. This is a 124 million parameter model from 2019, finetuned for an hour and a half on a CPU. It doesn't really know why the sky is blue, and it's very happy to make things up, with total confidence. The nanochat README says that talking to its speedrun model is a bit like talking to a kindergartener, and we're well below that. But the format is there, the turn taking is there, and as we'll see, on the narrow tasks that we specifically trained for, it can actually do quite well.

> **Exercise:** Re-run SFT with fewer SpellingBee examples, say five thousand instead of twenty thousand, and measure how often the model answers ordinary questions with "We are asked to find". What happens to its SpellingBee accuracy?

## The inference engine

Okay, so now let's talk about the Engine, which is what we've been using to sample from the model. In nanochat, the engine has two jobs. The first is to make sampling fast, and the second is to run the tools.

So first, why is naive sampling slow? Well, think about what we do in a naive sampling loop. We take the whole sequence, run it through the transformer, get the logits at the last position, sample a token, append it, and then run the whole sequence through the transformer again. But for all the tokens that we already processed, nothing changed! Their keys and values in every attention layer are exactly the same as last time, because of the causal mask, a token can never be affected by tokens that come after it. So we're recomputing the same things over and over, and for long sequences this gets really wasteful.

The fix is the KV cache. We keep the keys and values of every layer around, and when a new token comes in, we only compute the query, key and value for that one token, append the new key and value to the cache, and let the single new query attend to everything in the cache. Here's the KV cache in our gpt.py. It is tiny.

`gpt.py`
```python
class KVCache:
    """ Remembers the keys and values of all the tokens seen so far, for one layer. """

    def __init__(self):
        self.k = self.v = None

    def update(self, k, v):
        if self.k is not None:
            k = torch.cat([self.k, k], dim=2)
            v = torch.cat([self.v, v], dim=2)
        self.k, self.v = k, v
        return k, v

    def expand(self, batch_size):
        # prefill once with batch 1, then copy the cache for each of the samples
        self.k = self.k.expand(batch_size, -1, -1, -1)
        self.v = self.v.expand(batch_size, -1, -1, -1)
```

`gpt.py`
```python
class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.n_head = config.n_head

    def forward(self, x, kv_cache=None):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(C, dim=2)
        q, k, v = [t.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) for t in (q, k, v)] # (B, nh, T, hs)
        if kv_cache is not None:
            assert kv_cache.k is None or T == 1, "with a non-empty cache we feed in one token at a time"
            k, v = kv_cache.update(k, v) # k, v now also contain all the previous tokens
        y = F.scaled_dot_product_attention(q, k, v, is_causal=T > 1) # a single new token may look at everything
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)
```

So in the attention, if there's a cache, we append the new keys and values and get back the full keys and values. And one subtle thing: if we're feeding in a single token, we don't want a causal mask at all, because that one new token is allowed to look at everything before it. And in the model's forward, the positions have to start where the cache left off, otherwise the new token would get the position embedding of position zero. Let's make sure the cache actually gives exactly the same answer, and see how much faster it is.

In [21]:
import time
prompt = [tokenizer.bos] + tokenizer.encode("The meaning of life is")

t0 = time.time()
ids = torch.tensor([prompt])
with torch.no_grad():
    for _ in range(100):
        next_id = model(ids)[:, -1, :].argmax(dim=-1, keepdim=True) # recompute everything, every time
        ids = torch.cat([ids, next_id], dim=1)
t1 = time.time()
out, _ = engine.generate_batch(prompt, max_tokens=100, temperature=0.0)
t2 = time.time()
print(f"naive: {t1 - t0:.2f}s, with kv cache: {t2 - t1:.2f}s, same result: {ids[0].tolist() == out[0]}")

naive: 7.57s, with kv cache: 1.95s, same result: True


So with the KV cache, generating these 100 tokens is about four times faster on my CPU, and the result is exactly the same, token for token. And that gap only grows with length, because the naive version reprocesses the whole sequence for every new token, so the total work grows quadratically, while with the cache every new token costs about the same.

The other trick in the engine is for when we want many samples for the same prompt, which is going to be very important for reinforcement learning. We process the prompt once, with a batch size of one, and this is called the prefill. And then we just copy the cache for every sample, and then decode all the samples in parallel, as one batch.

Now the second job of the engine is tool use. And this is where it gets kind of cute. Let's look at the main loop.

`engine.py`
```python
# 3) decode, one token at a time, keeping track of the tool use state of each row
forced = [deque() for _ in range(num_samples)] # tokens we must insert (the tool output)
python_expr = [None] * num_samples # the tokens of the python expression being written
done = [False] * num_samples
max_tokens = min(max_tokens, model.config.block_size - len(tokens))
for _ in range(max_tokens):
    sampled = sample_next_token(logits, generator, temperature, top_k)[:, 0].tolist()
    column, masks = [], []
    for i in range(num_samples):
        is_forced = len(forced[i]) > 0
        token = forced[i].popleft() if is_forced else sampled[i]
        column.append(token)
        masks.append(0 if is_forced else 1)
        if token in stop_tokens:
            done[i] = True
        elif token == python_start:
            python_expr[i] = []
        elif token == python_end and python_expr[i] is not None:
            # the model wants to use the tool! run it and force its output into the stream
            result = use_calculator(self.tokenizer.decode(python_expr[i]))
            if result is not None:
                forced[i].extend([output_start, *self.tokenizer.encode(str(result)), output_end])
            python_expr[i] = None
        elif python_expr[i] is not None:
            python_expr[i].append(token)
```

> **Exercise:** Before I explain it, pause and think about it. The model is sampling tokens one at a time. At some point it emits python start, then some tokens, then python end. What exactly should the engine do at that point, and what should the model see next?

So every row keeps a little bit of state. When the model samples python start, we start recording the tokens of the expression. When it samples python end, we decode the expression, and evaluate it with our calculator. And then here's the trick: we put output start, the tokens of the result, and output end into a queue of forced tokens. And on the next steps, instead of using what the model sampled, we pop tokens from that queue. So the model sees the result of the tool appear in its own context, as if it wrote it, and then it continues sampling from there. And we also return a mask, one for sampled tokens and zero for forced tokens, and you'll see why that matters in reinforcement learning.

The calculator itself is deliberately very restricted. We only allow numbers and arithmetic, or the count method of a string, and we evaluate without any builtins. You really don't want to give a language model an unrestricted eval on your machine. In nanochat it's the same story, and there's a separate, more careful sandbox for running real python code.

`engine.py`
```python
def use_calculator(expr):
    """ A very restricted python "tool": arithmetic, and str.count (for counting letters) """
    expr = expr.replace(",", "")
    is_math = all(c in "0123456789*+-/.() " for c in expr) and "**" not in expr # no 9**9**9 please
    is_count = all(c.isalnum() or c in "'\"()._ " for c in expr) and ".count(" in expr and "__" not in expr
    if not (is_math or is_count):
        return None
    try:
        return eval(expr, {"__builtins__": {}}, {})
    except Exception:
        return None # the model wrote something broken, that's ok
```

Let's watch it in action. Here's a SpellingBee question from the test set, and I'm going to visualize what comes out of the engine, where green tokens were sampled by the model, and red tokens were forced in by the engine.

In [22]:
example = SpellingBee(size=10, split="test")[6]
print(example["messages"][0]["content"])
ids = tokenizer.render_for_completion(example)
out, masks = sft_engine.generate_batch(ids, max_tokens=256, temperature=0.0)
print(tokenizer.visualize(out[0][len(ids):], masks[0][len(ids):]))

How many u are in the word eurytherm


We| are| asked| to| find| the| number| '|u|'| in| the| word| '|e|ury|ther|m|'.| Let| me| try| a| manual| approach| first|.|
|
|First| spell| the| word| out|:|
|e|ury|ther|m|:|e|,|y|,|u|,|r|,|e|,|r|,|m|,|e|,|r|
|
|Then| count| the| occurrences| of| '|u|':|
|1|:|e|
|2|:|y|
|3|:|u| hit|!| count|=|1|
|4|:|r|
|5|:|e|
|6|:|r|
|7|:|m|
|8|:|e|
|9|:|r|
|
|This| gives| us| 1|.|
|
|Let| me| double| check| this| using| Python|:|

|<|python_start|>|'|e|ury|ther|m|'.|count|('|u|')|<|python_end|>|<|output_start|>|1|<|output_end|>|
|
|Python| gives| us| 1|.|
|
|My| final| answer| is|:|
|
|####| 1


So look at what happened here, because this is a really nice little example. The question is how many u's are in eurytherm. The model tries to spell it out, and it gets the spelling wrong. It writes e, y, u, r, and so on. It roughly knows which letters are in there, but not in the right order, because remember, it never actually sees the letters. Then it counts the u's in its own misspelled version, and it gets one, which happens to be right. And then it writes the python call, and in the python call it copies the word exactly, because copying a token is easy. Then the engine runs the code and forces in output start, one, output end, and those are the only three red tokens in the whole response. And Python says one, and the model reports one. So the tool is robust to the model's own sloppiness. Copying the word into python is a lot easier than spelling it.

## Evaluation

Okay so we have a model that seems to work, but "seems to work" is not good enough. We need numbers. And so in nanochat there's a chat_eval script, and there are basically two kinds of evaluations.

The first kind is categorical, which is for multiple choice. And here's a neat thing. We don't actually have to sample anything. We render the question, prime the assistant, run a single forward pass, and then we look at the logits at the very last position, but only at the logits of the allowed letters, A, B, C and D. And whichever one is highest, that's the model's answer.

`chat_eval.py`
```python
def run_categorical_eval(task, model, tokenizer, max_problems=None):
    """
    Multiple choice: we don't even need to sample. Run the prompt through the model once,
    look at the logits at the position where the assistant answers, and only compare the
    logits of the allowed letters. The highest one is the model's answer.
    """
    device = next(model.parameters()).device
    num_problems = len(task) if max_problems is None else min(len(task), max_problems)
    num_passed = 0
    for i in range(num_problems):
        conversation = task[i]
        ids = tokenizer.render_for_completion(conversation)
        logits = model(torch.tensor([ids], device=device))[0, -1] # (vocab_size,)
        letter_ids = [tokenizer.encode(letter)[0] for letter in conversation["letters"]]
        predicted = conversation["letters"][logits[letter_ids].argmax().item()]
        num_passed += task.evaluate(conversation, predicted)
    return num_passed / num_problems
```

So this is kind of generous to the model, because even if it would have said something else, we're forcing it to pick one of the letters. But that's how these evaluations are typically done, and it makes them fast and low-noise. And the random baseline for four choices is 25 percent.

The second kind is generative, for things like GSM8K and SpellingBee. Here we actually sample a response, then extract the number after the four hash signs, and compare it to the reference answer.

`chat_eval.py`
```python
def run_generative_eval(task, model, tokenizer, engine, max_problems=None, num_samples=1, temperature=0.0, max_tokens=256):
    """ Open ended: sample a response, parse out the answer, and check it. pass@k if num_samples > 1 """
    num_problems = len(task) if max_problems is None else min(len(task), max_problems)
    num_passed = 0
    for i in range(num_problems):
        conversation = task[i]
        ids = tokenizer.render_for_completion(conversation)
        results, _ = engine.generate_batch(ids, num_samples=num_samples, max_tokens=max_tokens, temperature=temperature, top_k=50)
        completions = [tokenizer.decode(r[len(ids):]) for r in results]
        num_passed += any(task.evaluate(conversation, c) for c in completions)
        print(f"\r{num_passed}/{i + 1} ({100 * num_passed / (i + 1):.2f}%)", end="", flush=True)
    print()
    return num_passed / num_problems
```

And then to get one number, nanochat has something called ChatCORE. It takes the accuracy on every task, rescales it so that zero means random guessing and one means perfect, and then averages them. So let's evaluate both the base model and our SFT model on 100 problems of each task. For the base model, I'm going to use the same chat format, which it has never seen, so we expect it to be pretty bad.

In [ ]:
!python chat_eval.py -i base -x 100
!python chat_eval.py -i sft -x 100

Output of the lecture's run:
```
$ python chat_eval.py -i base -x 100
ARC-Easy: 29.00%
ARC-Challenge: 26.00%
MMLU: 29.00%
GSM8K: 0.00%
SpellingBee: 0.00%
ChatCORE: 0.0240
$ python chat_eval.py -i sft -x 100
ARC-Easy: 24.00%
ARC-Challenge: 26.00%
MMLU: 26.00%
GSM8K: 0.00%
SpellingBee: 53.00%
ChatCORE: 0.1087
```

Okay so let's be honest about these numbers, because this is exactly where evaluations earn their keep. On the multiple choice tasks, both models are basically at chance, 25 percent. And with only 100 problems, the noise on each of these numbers is something like plus or minus four or five percent, so 24 versus 29 is not a real difference. GPT-2 small simply doesn't have the knowledge to answer MMLU questions, and SFT can't add knowledge that isn't there, it can only teach the format. You need a much bigger and better pretrained model for these to move, which is a big part of why nanochat pretrains its own, much stronger base models. GSM8K is zero for both. The SFT model has learned the format of a math solution, with calculator calls and the four hash signs at the end, but it can't actually do the reasoning. But SpellingBee goes from zero to 53 percent. That's a narrow, procedural task, and SFT taught the model a procedure, spell, count, double check with python, that it can actually execute. So the ChatCORE goes from 0.02 to 0.11, and essentially all of that comes from SpellingBee. And 53 percent is not great, but it's also not zero, which is exactly where we want to be for the next stage.

> **Exercise:** Try evaluating the base model on MMLU with a few-shot prompt instead of the chat format. How close can you get to the SFT model just by prompting?

## Reinforcement learning

Okay, so now we get to the last stage, and I think the most exciting one. Reinforcement learning.

So first, why do we need it? Karpathy has a really nice analogy for this. Think about how you learn from a textbook. Most of the textbook is exposition, background knowledge, and that's like pretraining. Then there are worked examples, where an expert shows you how to solve a problem step by step, and you imitate them. And that's SFT. But then at the end of every chapter, there are practice problems. You're given the problem, and the final answer is in the back of the book, but not the solution. And you have to try, and fail, and try again, and figure out what works for you. And that's reinforcement learning.

And here's the deep reason why this matters. When we do SFT on a GSM8K solution, we're asking the model to imitate a human's solution. But the human has a very different brain from the model. Some step that's trivial for the human might be a huge leap for the model, a leap that it can't make in a single token, and some step that the human spells out carefully might be totally obvious to the model. We don't actually know what the best token sequence is for this particular model to get to the right answer. And so the idea of reinforcement learning is, let's not tell it. Let the model try many times, and whatever works, do more of that.

So how do we actually do that? Let's say we have a question, and the model samples an answer y, and then we check the answer and give it a reward R, one if it's correct and zero if it's wrong. And what we want is to maximize the expected reward, the average reward over all the answers that the model might sample. Now, it turns out there's a really neat trick for computing the gradient of that. The gradient of the expected reward is equal to the expected value of the reward times the gradient of the log probability of the sample. This is called the policy gradient, or the REINFORCE trick, and it comes from the fact that the gradient of p is equal to p times the gradient of log p. And the log probability of a whole answer is just the sum of the log probabilities of its tokens, which is exactly the negative of the loss that our model already computes.

So in code, it's almost embarrassingly simple. Sample a bunch of answers. For each one, compute the sum of the log probabilities of its tokens, multiply by its reward, and do gradient ascent on that. And if you squint at it, it's basically SFT on your own samples, weighted by how good they were.

But there's one problem. If the rewards are zero or one, then the wrong answers get a weight of zero, so they're just ignored, and every right answer gets pushed up, even on easy questions where all the answers were right. So what we do is we subtract a baseline. For every question, we sample a group of answers, say sixteen of them, and the advantage of each answer is its reward minus the mean reward of the group. So if an answer is better than average for this question, its probability gets pushed up. If it's worse than average, its probability gets pushed down. And you can show that subtracting a baseline like this doesn't change the expected gradient at all, it just makes it a lot less noisy. This is the "group" in GRPO, group relative policy optimization.

Before we do this on a transformer, let's build some intuition on the tiniest possible example. Imagine a "language model" that only ever produces a single token, out of four options, and only option two is correct. The model is just four logits. We sample a group of sixteen attempts, reward the correct ones, compute the advantages, and do the policy gradient update.

In [23]:
def reinforce_toy(logits, steps=40):
    torch.manual_seed(0)
    logits = torch.tensor(logits, requires_grad=True)
    optimizer = torch.optim.SGD([logits], lr=1.0)
    for step in range(steps + 1):
        probs = logits.softmax(dim=0)
        samples = torch.multinomial(probs, num_samples=16, replacement=True) # a group of 16 attempts
        rewards = (samples == 2).float()                                     # only option 2 is correct
        advantages = rewards - rewards.mean()
        loss = -(probs.log()[samples] * advantages).mean()                   # the policy gradient loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step % 10 == 0:
            print(f"step {step:2d} | p(correct) {probs[2].item():.4f} | mean reward {rewards.mean().item():.3f}")

reinforce_toy([0.0, 0.0, 0.0, 0.0])

step  0 | p(correct) 0.2500 | mean reward 0.250
step 10 | p(correct) 0.8508 | mean reward 0.812
step 20 | p(correct) 0.9490 | mean reward 0.938
step 30 | p(correct) 0.9673 | mean reward 0.688
step 40 | p(correct) 0.9801 | mean reward 0.938


So the probability of the correct answer starts at a quarter, because the logits are all zero, and after only ten steps it's already at 85 percent, and it keeps climbing towards one. Nobody ever told it the answer. It just tried things, and did more of what worked. Now watch what happens if the model starts out really bad at this, if the correct answer is very unlikely to begin with.

In [24]:
reinforce_toy([0.0, 0.0, -10.0, 0.0])

step  0 | p(correct) 0.0000 | mean reward 0.000
step 10 | p(correct) 0.0000 | mean reward 0.000
step 20 | p(correct) 0.0000 | mean reward 0.000
step 30 | p(correct) 0.0000 | mean reward 0.000
step 40 | p(correct) 0.0000 | mean reward 0.000


Nothing happens. The correct answer is never sampled, so the reward is always zero, so all the advantages are zero, and there's no gradient at all. And this is a really important lesson about RL: it can only amplify behaviors that the model already does at least sometimes. It can't conjure up something from nothing. That's why we do SFT first, to get the model into a regime where it sometimes succeeds, and then RL takes it from there.

> **Exercise:** Pause here and try to write the loss yourself. You have the inputs and targets of a batch of sixteen sampled answers, the targets are minus one wherever we shouldn't train, and you have a tensor of sixteen advantages. Our model can return the per-token loss if you pass loss_reduction equals none.

Okay, let's look at the code. First, the rollouts.

`chat_rl.py`
```python
def get_batch():
    assistant_end = tokenizer.encode_special("<|assistant_end|>") # used for padding only, never trained on
    for step, example_idx in enumerate(itertools.cycle(range(len(train_task)))):
        conversation = train_task[example_idx]
        prompt = tokenizer.render_for_completion(conversation)
        with autocast:
            sequences, masks = engine.generate_batch(prompt, num_samples=args.num_samples, max_tokens=args.max_new_tokens,
                                                     temperature=args.temperature, top_k=args.top_k, seed=step)
        rewards = [train_task.reward(conversation, tokenizer.decode(seq[len(prompt):])) for seq in sequences]
        # pad the samples to the same length, then build inputs/targets just like in SFT
        max_length = max(len(seq) for seq in sequences)
        ids = torch.tensor([seq + [assistant_end] * (max_length - len(seq)) for seq in sequences], device=device)
        mask = torch.tensor([m + [0] * (max_length - len(m)) for m in masks], device=device)
        inputs, targets = ids[:, :-1], ids[:, 1:].clone()
        targets[mask[:, 1:] == 0] = -1 # don't train on the prompt, the tool outputs, or the padding
        rewards = torch.tensor(rewards, device=device)
        advantages = rewards - rewards.mean() # better than average => push up, worse => push down
        yield inputs, targets, rewards, advantages
```

So for every question, we render the prompt and use the engine to sample a whole group of answers in parallel, at temperature one, because we want diversity. We decode each one and get its reward from the task, and for SpellingBee and GSM8K that's just whether the answer after the hash signs is correct. Then we build inputs and targets exactly like in SFT, and here's where the engine's mask comes in: we set the targets to minus one for the prompt, for the padding, and for the tool outputs that were forced into the sequence. We only train on the tokens that the model actually chose. And then the advantages are the rewards minus their mean.

And here's the training step.

`chat_rl.py`
```python
for _ in range(args.examples_per_step):
    inputs, targets, rewards, advantages = next(batch_iterator)
    with autocast:
        logp = -model(inputs, targets, loss_reduction='none').view_as(inputs) # (B, T), 0 where target is -1
    # the policy gradient objective: the log probability of each sampled token, weighted by its advantage
    pg_obj = (logp * advantages.unsqueeze(-1)).sum()
    num_valid = (targets >= 0).sum().clamp(min=1)
    loss = -pg_obj / (num_valid * args.examples_per_step) # minimize the negative of what we want to maximize
    loss.backward()
    step_rewards.append(rewards.mean().item())
    step_lengths.append((targets >= 0).sum(dim=1).float().mean().item())
```

So the model gives us the per-token loss, which is the negative log probability, so we negate it to get the log probabilities, and the ignored tokens have a log probability of exactly zero. We multiply each token's log probability by the advantage of the answer it belongs to, sum it all up, and normalize by the number of valid tokens. And then the loss is the negative of that objective, because PyTorch minimizes and we want to maximize. And that's it. That's the whole algorithm.

Now, if you've read about GRPO or PPO, you might be wondering where all the other stuff is. And nanochat makes four deliberate simplifications, and they're written right at the top of the file.

`chat_rl.py`
```python
I put "GRPO" in quotes because, just like in nanochat, what we end up with is a lot simpler
and closer to plain old REINFORCE:
1) no KL regularization to a reference model (no trust region)
2) we are fully on-policy (one gradient step per batch of samples), so no PPO ratio + clip
3) token-level normalization of the loss (DAPO style), not sequence level
4) the advantage is just (reward - mean reward of the group), no division by the std
```

Number one, there's no KL penalty to a reference model. That's a regularizer that keeps the model from drifting too far from the SFT model, and in practice, for a short run like this, it's not really needed. Number two, there's no PPO ratio and clipping. Those exist because in PPO you take multiple gradient steps on the same batch of samples, so the samples become stale, off-policy, and you need to correct for that. But we take exactly one gradient step per batch, so we are always on-policy, and the ratio would always be exactly one. Number three, we normalize by the total number of tokens, not per answer, which is what the DAPO paper recommends, so that long answers aren't down-weighted. And number four, the advantage is just the reward minus the mean, we don't divide by the standard deviation. Put all of that together and what you're left with is honestly just good old REINFORCE, with a group mean baseline.

One more thing to notice. If all sixteen answers to a question are correct, or all of them are wrong, then all the advantages are zero, and that question contributes no gradient at all. So RL only learns from questions that are at the edge of the model's ability, where it sometimes succeeds and sometimes fails. If your model can never solve a problem, RL can't help you, and that's why we do SFT first.

Okay, so let's run it. We start from our SFT model, which gets about half of the SpellingBee questions right, so it's right at the edge of its ability, which is exactly where RL can help. On a GPU you'd use the defaults, sixteen questions per step and sixteen samples per question. On my CPU, I'm going to do 40 steps, with eight questions per step and eight samples each, so 64 attempts per step, and every ten steps we evaluate on 48 held-out questions, with eight samples each. This took about three hours, most of it spent sampling.

In [ ]:
!python chat_rl.py --task=SpellingBee --num-steps=40 --examples-per-step=8 --num-samples=8 --eval-examples=48 --lr=3e-5

Output of the lecture's run:
```
step    0 | eval pass@1 0.4427 | pass@8 0.9167
step    0 | reward 0.6406 | sampled tokens per answer 160.5 | lr 3.00e-05 | dt 193.4s
step    1 | reward 0.8750 | sampled tokens per answer 193.8 | lr 2.92e-05 | dt 349.8s
step    2 | reward 0.7812 | sampled tokens per answer 202.0 | lr 2.85e-05 | dt 248.2s
step    3 | reward 0.7344 | sampled tokens per answer 209.7 | lr 2.78e-05 | dt 252.6s
step    4 | reward 0.7812 | sampled tokens per answer 214.7 | lr 2.70e-05 | dt 252.2s
...
step   10 | eval pass@1 0.8385 | pass@8 1.0000
...
step   20 | eval pass@1 0.8672 | pass@8 0.9792
...
step   26 | reward 0.3438 | sampled tokens per answer 160.5 | lr 1.05e-05 | dt 153.6s
step   27 | reward 0.9844 | sampled tokens per answer 154.9 | lr 9.75e-06 | dt 135.3s
...
step   30 | eval pass@1 0.8776 | pass@8 1.0000
...
step   39 | reward 0.7500 | sampled tokens per answer 158.6 | lr 7.50e-07 | dt 142.3s
step   40 | eval pass@1 0.8750 | pass@8 1.0000
Saved the RL model to /root/.cache/nanochat-posttrain/chatrl.pt
```

So look at the eval. Before RL, pass at one is 44 percent. That's the fraction of individual samples, at temperature one, that are correct. But pass at eight is 92 percent: if we let the model try eight times, then for 92 percent of the questions at least one of the attempts is right. So the model already "knows" how to solve almost all of these, it's just unreliable. And then after only ten steps of RL, pass at one shoots up to 84 percent, and by the end it's at 87 and a half. And pass at eight goes to 100 percent. This is a really common pattern with RL on language models. A lot of what it does is take the things the model can do some of the time, and make it do them most of the time. It converts pass at k into pass at one.

The training reward on the other hand is really noisy, look at step 26, where it dips to 0.34. That's because every step only has eight questions, so some steps just happen to get harder words. Always look at a held out evaluation, not the training reward. And one more fun thing: the number of sampled tokens per answer goes from around 210 down to about 155. Nobody asked the model to be shorter. But a lot of the long answers were ones where it got lost, misspelling a word into a long garbage sequence of letters and never getting back on track, and those tend to get a reward of zero, so RL pushes them down.

And if we use our eval script, with greedy decoding on 100 test questions, SpellingBee goes from 53 percent for the SFT model to 86 percent after RL.

And now, finally, let's ask it the famous question. I'll take eight samples from both the SFT model and the RL model, and print their final answers.

In [25]:
from tasks import extract_answer
rl_model = load_model("rl", device).eval()
rl_engine = Engine(rl_model, tokenizer)

def answers(question, engine):
    ids = tokenizer.render_for_completion({"messages": [{"role": "user", "content": question}, {"role": "assistant", "content": ""}]})
    out, _ = engine.generate_batch(ids, num_samples=8, max_tokens=256, temperature=1.0, top_k=50)
    return [extract_answer(tokenizer.decode(o[len(ids):])) for o in out]

for question in ["How many r are in the word 'strawberry'?", "How many e are in cheese?", "How many r are in strawberry?"]:
    for name, engine in [("sft", sft_engine), ("rl", rl_engine)]:
        print(f"{name:3s} | {question:42s} | {answers(question, engine)}")

sft | How many r are in the word 'strawberry'?   | ['1', '3', '2', '2', '2', '2', '2', '1']


rl  | How many r are in the word 'strawberry'?   | ['3', '3', '3', '3', '3', '3', '3', '3']


sft | How many e are in cheese?                  | ['2', '5', '3', '2', '4', '3', '3', '3']


rl  | How many e are in cheese?                  | ['3', '3', '3', '3', '3', '3', '3', '3']


sft | How many r are in strawberry?              | ['2', '1', '1', '2', '2', '1', '2', '3']


rl  | How many r are in strawberry?              | ['4', '2', '3', '3', '1', '2', '2', '1']


Okay, so look at the first one. The SFT model gets three r's in strawberry only once out of eight tries, and the RL model gets it right eight out of eight times. Same thing for cheese, two out of eight before, eight out of eight after. That's RL.

But now look at the last one. "How many r are in strawberry", without the quotes and without "the word". And both models fall apart, the answers are all over the place. What's going on? Let's look at what the RL model actually writes.

In [26]:
print(chat("How many r are in strawberry?", engine=rl_engine, max_tokens=40))
print([tokenizer.decode([t]) for t in tokenizer.encode("How many r are in strawberry?")])
print([tokenizer.decode([t]) for t in tokenizer.encode("How many r are in the word 'strawberry'?")])

We are asked to find the number 'r' in the word 'sweet'. Let me try a manual approach first.

First spell the word out:
sweet:s,t
['How', ' many', ' r', ' are', ' in', ' strawberry', '?']
['How', ' many', ' r', ' are', ' in', ' the', ' word', " '", 'st', 'raw', 'berry', "'?"]


It says, we are asked to find the number r in the word "sweet". Sweet! It didn't even copy the right word. And the tokenization tells you exactly why. In the first version, the word is "space strawberry", a single token, one integer. In the version with quotes, it's "st", "raw", "berry", exactly the same tokens that the model has to write inside its quotes and in the python call. So in the quoted version, the model just has to copy three tokens, which is easy. But in the unquoted version, it has to know that the single token "space strawberry" is spelled s, t, r, a, w, and so on, and write it out as a completely different set of tokens. And it doesn't know that. So it reaches for something that's nearby in meaning, "sweet". Our SpellingBee training data mostly has rare words, which get split into several tokens either way, so the model never really had to learn how to unpack a single common token. And this is the tokenizer biting us, once again. If you remember the tokenization video, so many of the weird failures of LLMs come back to the tokenizer, and this is a perfect example.

So how would we fix this? Well, now that we understand the failure, we could add training data for exactly this: common words that are a single token. And that's how this stuff works in practice. You look at the failures, you figure out why they happen, and you go back and fix the data.

> **Exercise:** Look at the samples during RL. Does the model start trusting python over its manual count, or the other way around? Try changing the reward, for example a small penalty for long answers, and see what the model does. Be careful, RL will find any loophole in your reward.

## Scaling up, and what's next

Okay, so that's everything. Let's zoom out and recap. We took a base model, a document simulator, and we did three things. We defined a format for conversations, with special tokens and a loss mask. We did supervised finetuning on a mixture of conversations, where every dataset teaches something specific: chatting, multiple choice, math with a calculator, and spelling. And then we did reinforcement learning, where the model samples its own answers and we push up the ones that work. Along the way we wrote an inference engine with a KV cache and tool use, and evaluations to measure everything.

Now everything we wrote today maps one to one onto nanochat. Our tokenizer is nanochat's tokenizer.py, our tasks are its tasks directory, our engine is its engine.py, and our chat_sft, chat_rl, chat_eval and chat_cli are its scripts. The real thing has a few more bells and whistles. The base model is pretrained from scratch instead of GPT-2, and you can pick its size with a single depth parameter. It uses the Muon optimizer for the matrices, it runs in bfloat16 across eight GPUs with distributed data parallel, and it warm-starts the optimizer state from pretraining. And in nanochat, RL is done on GSM8K. But the ideas are exactly the ones we built today. If you have a GPU node, the whole thing is one script.

```bash
git clone https://github.com/karpathy/nanochat && cd nanochat
bash runs/speedrun.sh                                        # tokenizer, pretraining, SFT, eval (~2 hours on 8XH100)
torchrun --standalone --nproc_per_node=8 -m scripts.chat_rl  # reinforcement learning on GSM8K
python -m scripts.chat_cli -i rl                             # talk to it
```

And there's a lot that we didn't cover. We only did reinforcement learning on problems where we can check the answer automatically. But what about writing a poem, or a joke? There's no answer key for that. And that's where RLHF comes in, reinforcement learning from human feedback, where you train a separate reward model on human preferences, and then do RL against that reward model. Or methods like DPO, which skip the reward model entirely. But be careful, because a reward model is itself a neural network, and if you run RL against it for too long, the policy will find adversarial examples that fool it. So RLHF is a bit of a different beast from the kind of RL we did today, where the reward is the actual ground truth.

And there's the whole world of reasoning models, where you run RL for a very long time on hard math and code problems, and the model discovers on its own that it should think for a long time, try different approaches, backtrack, and double check its work. That's really what our SpellingBee task is a tiny cartoon of.

Okay, so that's it for today. I hope you enjoyed it. Take the code, run it, break it, add your own tasks, and see what your model learns. See you next time.